In [ ]:
import pandas as pd
import numpy as np
from torch import neg
from operator import pos

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, GridSearchCV, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB as nb
from xgboost import XGBClassifier
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score, log_loss
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import ConfusionMatrixDisplay

import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

# I/ Crossvalidation pour le choix des modèles de classification

## Modèles testés : 
    - Logistic Regression
    - Random Forest
    - XGBoost
    - HistGradientBoostingClassifier
    - CatBoostClassifier
    - Naive Bayes

### 1. Lecture des données et préparation des features

In [ ]:
original_train_dataset = pd.read_csv("conversion_data_train.csv")
original_train_dataset = original_train_dataset[original_train_dataset["age"]<= 100]

cvm_train_dataset = original_train_dataset.copy()

In [ ]:
X = cvm_train_dataset.drop(columns=["converted"])
y = cvm_train_dataset["converted"]

### 2. Split du dataset en train et test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### 3. Choix de la stratégie de cross-validation : StratifiedKFold (il permet de maintenir la proportion des classes dans chaque fold).

In [ ]:
cv = StratifiedKFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

### 4. Preprocessing pour les différents modèles (les modèles n'ayant pas tous les mêmes exigences en matière de données d'entrée).

In [ ]:
numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"]

# Preprocessing pour la régression logistique
preprocessor_linear = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features)
])

# Preprocessing pour Random Forest et XGBoost
preprocessor_trees = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

# Preprocessing pour Naive Bayes, catboost et HitsGradientBoosting 
preprocessor_cat = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

### 5. Instanciation des pipelines pour chaque modèle avec le préprocessing adapté.

In [ ]:
pipelines = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_linear),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_trees),
        ("model", RandomForestClassifier(random_state=42))
    ]),

    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_trees),
        ("model", XGBClassifier(random_state=42))
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor_cat),
        ("model", HistGradientBoostingClassifier(random_state=42))
    ]),

    "CatBoost": Pipeline([
        ("preprocessor", preprocessor_cat),
        ("model", CatBoostClassifier(
            verbose=0,
            random_seed=42,
        ))
    ]),
    "Naive Bayes": Pipeline([
        ("preprocessor", preprocessor_cat),
        ("model", nb())
    ])
}

### 6. Définition des métriques d'évaluation 

In [ ]:
scoring = {
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "pr_auc": "average_precision",
}

### 7. Boucle de crossvalidation pour chaque modèle, en utilisant les métriques de scoring définies.

In [ ]:
results = []

for name, pipeline_models in pipelines.items():

    scores = cross_validate(
        pipeline_models,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results.append({
        "Model": name,
        "F1": scores["test_f1"].mean(),
        "F1_std": scores["test_f1"].std(),
        "Recall": scores["test_recall"].mean(),
        "Precision": scores["test_precision"].mean(),
        "PR-AUC": scores["test_pr_auc"].mean(),
        "PR-AUC_std": scores["test_pr_auc"].std()
    })


df_results = pd.DataFrame(results)

### 8. Affichage des résultats de crossvalidation pour chaque modèle et chaque métrique.

In [ ]:
display(
    df_results.sort_values(
        by="F1",
        ascending=False
    )
)

# III/ Cross validation des entrainements des modèles Logistic Regression, HistGradientBoostingClassifier et XGBoost afin de trouver les meilleurs hyperparamètres pour chaque modèle.

In [ ]:
cv = StratifiedKFold( 
    n_splits=5,
    shuffle=True,
    random_state=42
)

#### Modèle 1 : HistGradientBoostingClassifier

In [ ]:
# preprocessing HistGradientBoostingClassifier

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"]

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Gridsearch 1 qui teste l'apprentissage du modèle : 
params_hgbc_gs1 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__max_iter": [100, 300, 500],
}

# Gridsearch 2 qui teste la profondeur et le surapprentissage du modèle :
params_hgbc_gs2 = {
    "model__max_depth": [1, 3, 5, 7],
    "model__min_samples_leaf": [1, 5, 10],
    "model__l2_regularization": [0.0, 0.5, 1.0],
}

# Gridsearch 3 qui teste l'interaction entre les hyperparamètres du modèle :
params_hgbc_gs3 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__max_iter": [100, 300, 500],
    "model__max_depth": [1, 3, 5, 7],
    "model__min_samples_leaf": [1, 5, 10],
    "model__l2_regularization": [0.0, 0.5, 1.0],
}

In [ ]:
model = HistGradientBoostingClassifier(random_state=42)

pipeline_hgbc = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
    "log_loss": "neg_log_loss",
}

In [ ]:
grid_hgbc_gs1 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs1,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs1.fit(X_train, y_train)

grid_hgbc_gs2 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs2,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs2.fit(X_train, y_train)

grid_hgbc_gs3 = GridSearchCV(
    estimator=pipeline_hgbc,
    param_grid=params_hgbc_gs3,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_hgbc_gs3.fit(X_train, y_train)

In [ ]:
results_hgbc_gs1 = pd.DataFrame(grid_hgbc_gs1.cv_results_)
display(results_hgbc_gs1.head(10))

In [ ]:
results_hgbc_gs2 = pd.DataFrame(grid_hgbc_gs2.cv_results_)
display(results_hgbc_gs2.head(10))

In [ ]:
results_hgbc_gs3 = pd.DataFrame(grid_hgbc_gs3.cv_results_)
display(results_hgbc_gs3.head(10))

In [ ]:
def summarize_grid_hgbc(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)

    columns = [
        "param_model__learning_rate",
        "param_model__max_iter",
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__l2_regularization",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_pr_auc",
        "rank_test_f1",
        "rank_test_pr_auc",
        "rank_test_log_loss"
    ]

    # Certaines colonnes n'existent pas pour tous les modèles
    columns = [column for column in columns if column in results.columns]

    print(f"\n===== {model_name} =====")
    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print(f"Meilleur F1 moyen : {grid.best_score_:.4f}")
    print(f"Meilleur PR-AUC moyen : {results.loc[grid.best_index_, 'mean_test_pr_auc']:.4f}")
    print(f"Meilleur Log Loss moyen : {-results.loc[grid.best_index_, 'mean_test_log_loss']:.4f}")
    print(f"Meilleur Recall moyen : {results.loc[grid.best_index_, 'mean_test_recall']:.4f}")

    return results[columns].sort_values("rank_test_f1")


results_hgbc_gs1 = summarize_grid_hgbc(grid_hgbc_gs1, "Gradient Boosting Classifier - Grid Search 1")
results_hgbc_gs2 = summarize_grid_hgbc(grid_hgbc_gs2, "Gradient Boosting Classifier - Grid Search 2")
results_hgbc_gs3 = summarize_grid_hgbc(grid_hgbc_gs3, "Gradient Boosting Classifier - Grid Search 3")

display(results_hgbc_gs1.head(10))
display(results_hgbc_gs2.head(10))
display(results_hgbc_gs3.head(10))

In [ ]:
model_scores = pd.DataFrame({
    "model": ["HGBG GS1", "HGBG GS2", "HGBG GS3"],
    "mean_cv_f1": [
        grid_hgbc_gs1.best_score_,
        grid_hgbc_gs2.best_score_,
        grid_hgbc_gs3.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "HGBG GS1": grid_hgbc_gs1,
    "HGBG GS2": grid_hgbc_gs2,
    "HGBG GS3": grid_hgbc_gs3,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")

#### Les hyperparamètres retenus pour le modèle de HistGradientBoostingClassifier sont les suivants :
- learning_rate : 
- max_depth : 
- max_leaf_nodes :
- min_samples_leaf :
- max_iter :
- L2 regularization :

#### Modèle 2 : Logistic Regression

In [ ]:
numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"] # new_user est à exclure car déjà un int64 ? 

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Gridsearch régularisation elasticnet : 
params_lr_elasticnet = {
    "model__penalty": ["elasticnet"],
    "model__l1_ratio": [0.0, 0.5, 1.0],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
}

# Gridsearch régularisation Ridge :
params_lr_ridge = {
    "model__penalty": ["l2"],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga", "lbfgs", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
    }

# Gridsearch régularisation Lasso :
params_lr_lasso = {
    "model__penalty": ["l1"],
    "model__max_iter": [300, 500, 1000],
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__solver": ["saga", "liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__tol": [1e-4, 1e-3, 1e-2]
    }

In [ ]:
model = LogisticRegression(random_state=42)

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
}

In [ ]:
grid_lr_lasso = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_lasso,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_lasso.fit(X_train, y_train)

grid_lr_ridge = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_ridge,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_ridge.fit(X_train, y_train)

grid_lr_elasticnet = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=params_lr_elasticnet,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_lr_elasticnet.fit(X_train, y_train)

In [ ]:
results_lr_lasso = pd.DataFrame(grid_lr_lasso.cv_results_)
display(results_lr_lasso.head(10))

In [ ]:
results_lr_ridge = pd.DataFrame(grid_lr_ridge.cv_results_)
display(results_lr_ridge.head(10))

In [ ]:
results_lr_elasticnet = pd.DataFrame(grid_lr_elasticnet.cv_results_)
display(results_lr_elasticnet.head(10))

In [ ]:
def summarize_grid_lr(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)

    columns = [
        "param_model__penalty",
        "param_model__max_iter",
        "param_model__C",
        "param_model__l1_ratio",
        "param_model__solver",
        "param_model__class_weight",
        "param_model__tol",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_pr_auc",
        "rank_test_f1",
        "rank_test_pr_auc",
    ]

    # Certaines colonnes n'existent pas pour tous les modèles
    columns = [column for column in columns if column in results.columns]

    print(f"\n===== {model_name} =====")
    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print(f"Meilleur F1 moyen : {grid.best_score_:.4f}")
    print(f"Meilleur PR-AUC moyen : {results.loc[grid.best_index_, 'mean_test_pr_auc']:.4f}")
    print(f"Meilleur Recall moyen : {results.loc[grid.best_index_, 'mean_test_recall']:.4f}")

    return results[columns].sort_values("rank_test_f1")


results_lr_lasso = summarize_grid_lr(grid_lr_lasso, "Logistic Regression Lasso - Grid Search")
results_lr_ridge = summarize_grid_lr(grid_lr_ridge, "Logistic Regression Ridge - Grid Search")
results_lr_elasticnet = summarize_grid_lr(grid_lr_elasticnet, "Logistic Regression ElasticNet - Grid Search")

display(results_lr_lasso.head(10))
display(results_lr_ridge.head(10))
display(results_lr_elasticnet.head(10))

In [ ]:
model_scores = pd.DataFrame({
    "model": ["Logitic Regression - Lasso", "Logitic Regression - Ridge", "Logitic Regression - ElasticNet"],
    "mean_cv_f1": [
        grid_lr_lasso.best_score_,
        grid_lr_ridge.best_score_,
        grid_lr_elasticnet.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "Logistic Regression - Lasso": grid_lr_lasso,
    "Logistic Regression - Ridge": grid_lr_ridge,
    "Logistic Regression - ElasticNet": grid_lr_elasticnet,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")

#### Les hyperparamètres retenus pour le modèle de Régression Logistique sont les suivants :
- Penalty: 
- C :
- l1_ratio :
- solver :
- class_weight :
- tol :

#### Modèle 3 : XGBoost

In [ ]:
# Preprocessing XGBoost

numeric_features = ["age", "total_pages_visited"]
categorical_features = ["country", "source", "new_user"]

categorical_transformer = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

scale_pos_weight = n_neg / n_pos

print(f"Classe 0 : {n_neg}")
print(f"Classe 1 : {n_pos}")
print(f"scale_pos_weight de référence : {scale_pos_weight:.3f}")

In [ ]:
# Gridsearch 1 qui teste l'apprentissage du modèle :
params_xgb_gs1 = {
    "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
    "model__n_estimators": [100, 300, 500],
}

# Gridsearch 2 qui teste la profondeur et le surapprentissage du modèle :
params_xgb_gs2 = {
    "model__max_depth": [1, 3, 5, 7],
    "model__min_child_weight": [1, 3, 5],
    "model__colsample_bytree": [0.5, 0.7, 1.0],
    "model__reg_alpha": [0, 0.5, 1.0],
    "model__subsample": [0.5, 0.7, 1.0],
    
}

# Gridsearch 3 qui teste l'interaction entre les hyperparamètres du modèle :
params_xgb_gs3 = {
    "model__scale_pos_weight": [
        1.0,
        scale_pos_weight / 2,
        scale_pos_weight,
        scale_pos_weight * 1.5,
    ],
}

In [ ]:
model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=1,
)

pipeline_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

scoring = {
    "f1": "f1",
    "recall": "recall",
    "pr_auc": "average_precision",
    "log_loss": "neg_log_loss",
}

In [ ]:
grid_xgb_gs1 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs1,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs1.fit(X_train, y_train)

grid_xgb_gs2 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs2,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs2.fit(X_train, y_train)

grid_xgb_gs3 = GridSearchCV(
    estimator=pipeline_xgb,
    param_grid=params_xgb_gs3,
    scoring=scoring,
    refit="pr_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_xgb_gs3.fit(X_train, y_train)

In [ ]:
# Filtrer avec ls 3 Gridsearch précédents avant de lancer le GS4 
# afin d'éviter un nombre d'entrainements trop importants.

# # Gridsearch 4 qui teste l'interaction entre les hyperparamètres du modèle :
# params_xgb_gs4 = {
#     "model__learning_rate": [0.01, 0.1, 0.3, 0.5],
#     "model__n_estimators": [100, 300, 500],
#     "model__max_depth": [1, 3, 5, 7],
#     "model__min_child_weight": [1, 3, 5],
#     "model__l2_regularization": [0.0, 0.5, 1.0],
#     "model__scale_pos_weight": [1, scale_pos_weight],
#     "model__colsample_bytree": [0.5, 0.7, 1.0],
#     "model__reg_alpha": [0, 0.5, 1.0],
# }

# grid_xgb_gs4 = GridSearchCV(
#     estimator=pipeline_xgb,
#     param_grid=params_xgb_gs4,
#     scoring=scoring,
#     refit="pr_auc",
#     cv=cv,
#     n_jobs=-1,
#     return_train_score=True
# )
# grid_xgb_gs3.fit(X_train, y_train)

In [ ]:
results_xgb_gs1 = pd.DataFrame(grid_xgb_gs1.cv_results_)
display(results_xgb_gs1.head(10))

In [ ]:
results_xgb_gs2 = pd.DataFrame(grid_xgb_gs2.cv_results_)
display(results_xgb_gs2.head(10))

In [ ]:
results_xgb_gs3 = pd.DataFrame(grid_xgb_gs3.cv_results_)
display(results_xgb_gs3.head(10))

In [ ]:
def summarize_grid_xgb(grid, model_name):
    results = pd.DataFrame(grid.cv_results_)

    columns = [
        "param_model__learning_rate",
        "param_model__max_iter",
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__l2_regularization",
        "mean_test_f1",
        "std_test_f1",
        "mean_test_recall",
        "mean_test_pr_auc",
        "rank_test_f1",
        "rank_test_pr_auc",
        "rank_test_log_loss"
    ]

    # Certaines colonnes n'existent pas pour tous les modèles
    columns = [column for column in columns if column in results.columns]

    print(f"\n===== {model_name} =====")
    print("Meilleurs paramètres :")
    print(grid.best_params_)
    print(f"Meilleur F1 moyen : {grid.best_score_:.4f}")
    print(f"Meilleur PR-AUC moyen : {results.loc[grid.best_index_, 'mean_test_pr_auc']:.4f}")
    print(f"Meilleur Log Loss moyen : {-results.loc[grid.best_index_, 'mean_test_log_loss']:.4f}")
    print(f"Meilleur Recall moyen : {results.loc[grid.best_index_, 'mean_test_recall']:.4f}")

    return results[columns].sort_values("rank_test_f1")


results_xgb_gs1 = summarize_grid_xgb(grid_xgb_gs1, "XGBoost Classifier - Grid Search 1")
results_xgb_gs2 = summarize_grid_xgb(grid_xgb_gs2, "XGBoost Classifier - Grid Search 2")
results_xgb_gs3 = summarize_grid_xgb(grid_xgb_gs3, "XGBoost Classifier - Grid Search 3")

display(results_xgb_gs1.head(10))
display(results_xgb_gs2.head(10))
display(results_xgb_gs3.head(10))

In [ ]:
model_scores = pd.DataFrame({
    "model": ["XGB GS1", "XGB GS2", "XGB GS3"],
    "mean_cv_f1": [
        grid_xgb_gs1.best_score_,
        grid_xgb_gs2.best_score_,
        grid_xgb_gs3.best_score_,
    ],
})

display(model_scores.sort_values("mean_cv_f1", ascending=False))

grids = {
    "XGB GS1": grid_xgb_gs1,
    "XGB GS2": grid_xgb_gs2,
    "XGB GS3": grid_xgb_gs3,
}

best_model_name = max(
    grids,
    key=lambda name: grids[name].best_score_
)

best_grid = grids[best_model_name]
best_model = best_grid.best_estimator_

print(f"Modèle sélectionné : {best_model_name}")
print(f"F1 moyen en validation croisée : {best_grid.best_score_:.4f}")
print(f"Meilleurs paramètres : {best_grid.best_params_}")